In [ ]:
import os
os.environ["SHAM_V2_STAGE"] = "tools"
os.environ["SHAM_V2_DONOR"] = "HuggingFaceTB/SmolLM3-3B"
os.environ["SHAM_V2_DATASET"] = "sham-v2-candidate-checkpoint"
os.environ.setdefault("SHAM_V2_DTYPE", "bfloat16")


# شام v2 — مرحلة الأدوات (الأدوات الـ49 ومسارات عملها)


In [ ]:
# SHAM_SECRETS_FALLBACK_V3 (2026-10-15)
# بعد ساعات من التدريب فشلت خلية الحفظ بـ NameError: '_sf_orig' / '_sf_creds'
# (الغلاف القديم V1 يعتمد على متغيرات عامة اختفت). هذه الخلية مستقلة تماماً:
# لا تعتمد على أي متغير عام، تضبط KAGGLE_USERNAME/KAGGLE_KEY في البيئة و
# ~/.kaggle/kaggle.json فوراً، وتلتقط أي خطأ من الأغلفة السابقة (حتى NameError)
# فترجع القيم من البيئة. آمنة إن أُعيد تشغيلها.
def _sham_secrets_fallback_v3():
    import glob, json, os

    def load_creds(_glob=glob, _json=json, _os=os):
        u, k = _os.environ.get("KAGGLE_USERNAME"), _os.environ.get("KAGGLE_KEY") or _os.environ.get("KAGGLE_API_TOKEN")
        if u and k:
            return {"username": u, "key": k}
        pats = ["/kaggle/input/sham-runtime-creds/**/kaggle.json", "/kaggle/input/**/kaggle.json",
                _os.path.expanduser("~/.kaggle/kaggle.json")]
        for pat in pats:
            for p in sorted(_glob.glob(pat, recursive=True)):
                try:
                    with open(p) as f:
                        d = _json.load(f)
                    if d.get("username") and d.get("key"):
                        return d
                except Exception:
                    pass
        return {}

    c = load_creds()
    if c:
        os.environ.setdefault("KAGGLE_USERNAME", c["username"])
        if not (os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")):
            os.environ["KAGGLE_KEY"] = c["key"]
            if str(c["key"]).startswith("KGAT_"):
                os.environ["KAGGLE_API_TOKEN"] = c["key"]
        try:
            kp = os.path.expanduser("~/.kaggle/kaggle.json")
            if not os.path.exists(kp):
                os.makedirs(os.path.dirname(kp), exist_ok=True)
                with open(kp, "w") as f:
                    json.dump({"username": c["username"], "key": c["key"]}, f)
                os.chmod(kp, 0o600)
        except Exception:
            pass
    print("SHAM_SECRETS_FALLBACK_V3: بيانات كاكل", "جاهزة في البيئة" if c else "غير موجودة")
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe3", False):
        return

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _load=load_creds, **k):
        try:
            return _prev(self, label, *a, **k)
        except BaseException as e:
            if isinstance(e, KeyboardInterrupt):
                raise
            if label == "GITHUB_TOKEN":
                return ""
            if label in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                d = _load()
                if d:
                    return d["username"] if label == "KAGGLE_USERNAME" else d["key"]
            raise

    get_secret._sham_safe3 = True
    get_secret._sham_safe2 = True  # يمنع V2 من إعادة التغليف لو أعيد تشغيلها
    get_secret._sham_safe = True   # ويمنع V1 كذلك
    UserSecretsClient.get_secret = get_secret

try:
    _sham_secrets_fallback_v3()
except Exception as _sf3_exc:
    print("SHAM_SECRETS_FALLBACK_V3 غير مفعّل:", type(_sf3_exc).__name__)


In [ ]:
# SHAM_QUIET_START_V4 (2026-10-15)
# المالك لاحظ أن اللون الأصفر/البرتقالي للدفتر يظهر حين يكتب الدفتر أخطاء/تحذيرات
# (stderr) ثم يفشل لاحقاً. هذه الخلية تمنع الأخطاء المتوقعة من مصدرها، ولا تغيّر
# أي إعداد للنموذج: (1) تتذكّر السر الذي رفضه كاكل (HTTP 400) فلا تعيد طلبه وتطبع
# الخطأ كل مرة، وتعطي KAGGLE_* من البيئة مباشرة دون طلب الخادم، (2) تضبط
# __main__.__spec__ = None حتى يعمل multiprocessing(spawn) بلا AttributeError،
# (3) تخفّض تحذيرات HuggingFace المتوقعة (طلبات بلا توكن، أشرطة التقدم).
def _sham_quiet_start_v4():
    import os, sys, logging, warnings
    m = sys.modules.get("__main__")
    if m is not None and getattr(m, "__spec__", "missing") == "missing":
        try:
            m.__spec__ = None
        except Exception:
            pass
    os.environ.setdefault("HF_HUB_VERBOSITY", "error")
    os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
    os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*")
    for n in ("huggingface_hub", "datasets", "fsspec", "urllib3"):
        logging.getLogger(n).setLevel(logging.ERROR)
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    if getattr(UserSecretsClient.get_secret, "_sham_safe4", False):
        return
    failed = set()

    def get_secret(self, label, *a, _prev=UserSecretsClient.get_secret, _failed=failed, **k):
        env = {"KAGGLE_USERNAME": os.environ.get("KAGGLE_USERNAME"),
               "KAGGLE_KEY": os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN")}
        if label in env and env[label]:
            return env[label]            # جاهزة من V3 / sham-runtime-creds: لا طلب ولا خطأ 400
        if label in _failed and label == "GITHUB_TOKEN":
            return ""
        try:
            return _prev(self, label, *a, **k)
        except KeyboardInterrupt:
            raise
        except BaseException:
            _failed.add(label)
            if label == "GITHUB_TOKEN":
                return ""
            raise

    for attr in ("_sham_safe4", "_sham_safe3", "_sham_safe2", "_sham_safe"):
        setattr(get_secret, attr, True)
    UserSecretsClient.get_secret = get_secret

try:
    _sham_quiet_start_v4()
    print("SHAM_QUIET_START_V4: مفعّل")
except Exception as _q4_exc:
    print("SHAM_QUIET_START_V4 غير مفعّل:", type(_q4_exc).__name__)


### 1) سحب الكود


In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


### 2) المكتبات


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "tokenizers", "soundfile", "safetensors", "transformers", "datasets", "accelerate", "huggingface_hub", "pillow"], check=True)
print("المكتبات جاهزة.")

### 3) الإعداد: مفردات المانح


In [ ]:
import time
SESSION_START = time.time()
import os, json, shutil, random
import sham_v2_train as V
DONOR = os.environ.get("SHAM_V2_DONOR", V.DEFAULT_DONOR)
V.setup_env(V.donor_text_vocab(DONOR))          # SHAM_TEXT_VOCAB (the donor's own text width): يجب أن يسبق استيراد model.py
import torch
from pathlib import Path
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"الجهاز: {device}")
from huggingface_hub import hf_hub_download
from text_tokenizer import ShamTextTokenizer
tokenizer = ShamTextTokenizer.load(hf_hub_download(DONOR, "tokenizer.json"))
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")
print(f"مُرمِّز المانح: {tokenizer.vocab_size}")

### 4) النموذج


In [ ]:
from sham_inputs import fetch_dataset, publish_dataset
V_DS = V.DEFAULT_DATASET
OWN = fetch_dataset(V_DS)
_own = sorted(OWN.rglob("final_merged.pt")) if OWN else []
model, start_step = V.prepare_model(DONOR, _own[0] if _own else None, device)
STAGE2_OWN = None
print(f"صفوف الوسائط والرموز الخاصة القابلة للتدريب: {V.freeze_for_media(model):,} (الباقي مجمّد)")

### 5) قياس قبل الجلسة (أدوات، خسارة محجوزة، بت/بايت)


In [ ]:
N_TOOL_CASES = int(os.environ.get("SHAM_V2_TOOL_CASES", "100" if device == "cuda" else "40"))
tools_before = V.measure_tools(model, tokenizer, n_cases=N_TOOL_CASES)
print(f"أدوات قبل: صحيح {tools_before['right']}/{tools_before['cases']} | استدعاء سليم {tools_before['valid']} | ask_user {tools_before['asked']}")# the merge gate's yardstick: the same fixed held-out text before and after (Sham v1's rule: nothing replaces the served checkpoint unless it is measured not worse)
HELDOUT = V.heldout_text()
loss_before = V.heldout_loss(model, tokenizer, device, HELDOUT)
print(f"خسارة النص المحجوز قبل: {loss_before:.4f}")
HDOCS = V.heldout_docs()
TV = int(os.environ["SHAM_TEXT_VOCAB"])
bpb_before = {k: V.heldout_bpb(model, tokenizer, device, v, TV) for k, v in HDOCS.items()}
print(f"بت/بايت قبل: عربي {bpb_before['ar']:.4f} | إنجليزي {bpb_before['en']:.4f}")


### 6) بيانات الأدوات


In [ ]:
SEED = int(time.time()) // 86400
mix = V.tools_stage_batches(tokenizer, SEED, int(os.environ.get("SHAM_V2_TOOL_EXAMPLES", "4000")), 2, V.arabic_replay_pairs(SEED, 600))
print(f"مرحلة الأدوات: {len(mix):,} دفعة (كل أداة تُستدعى ٣ مرات على الأقل + مسارات العمل + الهوية + إعادة عربية)")


### 7) التدريب


In [ ]:
model.to(device)
HOURS = float(os.environ.get("SHAM_V2_HOURS", "2.0" if device == "cuda" else "6.0"))
HOURS = min(HOURS, max(0.5, (11.0 * 3600 - (time.time() - SESSION_START) - 40 * 60) / 3600))
losses, final_step = V.run_session(model, start_step, mix, device, HOURS)
print(f"انتهى: {len(losses):,} خطوة — الخطوة النهائية {final_step:,}")
if losses:
    print(f"الخسارة: أول 10 = {sum(losses[:10]) / min(10, len(losses)):.4f} | آخر 10 = {sum(losses[-10:]) / min(10, len(losses)):.4f}")


### بعد الجلسة: قياس الأدوات


In [ ]:
tools_after = V.measure_tools(model, tokenizer, n_cases=N_TOOL_CASES)
print(f"أدوات بعد: صحيح {tools_after['right']}/{tools_after['cases']} | استدعاء سليم {tools_after['valid']} | ask_user {tools_after['asked']}")
for r in tools_after["rows"][:6]:
    print("  ", r)

### الحفظ ثم البوابة ثم النشر (لا نشر إن ساءت القياسات أو نشر دفتر آخر أثناء الجلسة)


In [ ]:
from checkpoint import save_checkpoint
from sham_tok_id import stamp, resolve
import json as _json
upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    shutil.rmtree(upload_dir)
upload_dir.mkdir(parents=True)
extra = {"native_tokenizer": True, **stamp(tokenizer)}
save_checkpoint(upload_dir / "final_merged.pt", model, final_step, extra=extra)
shutil.copy2("/kaggle/working/sham_small_tokenizer.json", upload_dir / "sham_small_tokenizer.json")
for _f in ("image_tokenizer.pt", "audio_tokenizer.pt"):
    if (Path("/kaggle/working") / _f).is_file():                   # a tools-only session never builds the media tokenizers
        shutil.copy2(Path("/kaggle/working") / _f, upload_dir / _f)
for _ledger in Path("/kaggle/working/checkpoints").glob("*_ledger.json"):
    shutil.copy2(_ledger, upload_dir / _ledger.name)
_saved = torch.load(upload_dir / "final_merged.pt", map_location="cpu", weights_only=False)
_path, _note = resolve(_saved.get("extra"), upload_dir / "sham_small_tokenizer.json")
assert _path == str(upload_dir / "sham_small_tokenizer.json") and _note.startswith("ok"), _note        # serve.py would accept it
del _saved
_hist = []
if OWN:
    _hp = sorted(OWN.rglob("v2_history.json"))
    if _hp:
        _hist = _json.loads(_hp[0].read_text(encoding="utf-8"))
_hist.append({"step": final_step, "stage": V.STAGE, "device": device, "donor": DONOR, "tools_before": {k: v for k, v in tools_before.items() if k != "rows"},
              "tools_after": {k: v for k, v in tools_after.items() if k != "rows"}, "train_loss_last": losses[-20:] if losses else []})
loss_after = V.heldout_loss(model, tokenizer, device, HELDOUT)
verdict = V.gate(loss_before, loss_after, {k: v for k, v in tools_before.items() if k != "rows"}, {k: v for k, v in tools_after.items() if k != "rows"})
print(f"خسارة النص المحجوز بعد: {loss_after:.4f} | بوابة الدمج: {'مقبول' if verdict['ok'] else 'مرفوض: ' + '; '.join(verdict['reasons'])}")
bpb_after = {k: V.heldout_bpb(model, tokenizer, device, v, TV) for k, v in HDOCS.items()}
print(f"بت/بايت بعد: عربي {bpb_after['ar']:.4f} | إنجليزي {bpb_after['en']:.4f}")
_hist[-1]["gate"] = verdict
_hist[-1]["bpb_before"], _hist[-1]["bpb_after"] = bpb_before, bpb_after
_hist[-1]["dtype"] = str(next(model.parameters()).dtype)
_hist[-1]["resident_gb"] = round(sum(p.numel() * p.element_size() for p in model.parameters()) / 1e9, 2)
(upload_dir / "v2_history.json").write_text(_json.dumps(_hist, ensure_ascii=False, default=float), encoding="utf-8")
print("بصمة المُرمِّز:", _note)
_moved = V.published_step(fetch_dataset, V_DS) > start_step               # another notebook of this lineage published while this one trained: do not overwrite it
if verdict["ok"] and not _moved:
    publish_dataset(upload_dir, V_DS, f"sham v2 at step {final_step:,}")
elif _moved:
    print("لم يُنشر: دفتر آخر من السلالة نفسها نشر نقطة أحدث أثناء هذه الجلسة؛ لا أكتب فوقها.")
else:
    print("لم يُنشر: النقطة الجديدة لا تجتاز البوابة؛ تبقى النقطة المخدومة كما هي (الخطوة السابقة).")

In [ ]:
# Ask the restarter to run right after this session ends (added 2026-10-16; deployment only, no training change).
try:
    import sham_trigger_restarter
    sham_trigger_restarter.trigger()
except Exception as _exc:
    print(f"sham_trigger_restarter skipped: {_exc}")
